# ImageNet-1K formal training: VisionLLaMA + LSSO/RRLSSO

Google Colab / Colab Enterprise entry point for one RTX PRO 6000 Blackwell 96GB. Run cells from top to bottom. Training launches as a detached process, monitoring is an independent 120-second heartbeat, checkpoints live in a shallow `/content/LSSO-checkpoints` directory, and a separate archive cell creates a validated resume bundle.

## 1. Get the repository
Edit only `REPO_URL` if your remote differs, then click the cell. The branch must already be pushed to the remote.

In [ ]:
from pathlib import Path
import os, sys, subprocess, getpass, json, shutil, signal, time

REPO_URL = 'https://github.com/Yang916-yy/LSSO.git'
BRANCH = 'experiment/vision-llama-bidirectional'
COLAB_ROOT = Path('/content') if Path('/content').is_dir() else Path.home()
ROOT = COLAB_ROOT / 'LSSO'
if not (ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(ROOT)
print('repository:', ROOT)
print('commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# Keep PyTorch on the same CUDA major/minor as Colab's nvcc. MathDx uses
# device LTO, so CUDA 12 and CUDA 13 build components cannot be mixed.
nvcc = shutil.which('nvcc')
assert nvcc, 'The CUDA toolkit/nvcc is missing from this runtime'
nvcc_version = subprocess.check_output([nvcc, '--version'], text=True)
torch_cuda = subprocess.check_output(
    [sys.executable, '-c', 'import torch; print(torch.version.cuda or \"none\")'], text=True
).strip()
if 'release 12.8' in nvcc_version:
    wanted_cuda = '12.8'
    torch_packages = ('torch==2.11.0', 'torchvision==0.26.0')
    torch_index = 'https://download.pytorch.org/whl/cu128'
elif 'release 13.' in nvcc_version:
    wanted_cuda = '13.0'
    torch_packages = ('torch==2.12.1', 'torchvision==0.27.1')
    torch_index = 'https://download.pytorch.org/whl/cu130'
else:
    raise RuntimeError(f'Expected CUDA Toolkit 12.8 or 13.x, got:\n{nvcc_version}')
if not torch_cuda.startswith(wanted_cuda):
    assert 'torch' not in sys.modules, 'Restart the runtime, then run from the first cell'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    *torch_packages, '--index-url', torch_index], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'cmake>=3.26'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[experiments]'], cwd=ROOT, check=True)
print('dependencies installed; PyTorch CUDA:',
      subprocess.check_output([sys.executable, '-c', 'import torch; print(torch.version.cuda)'], text=True).strip())

## 2. Authenticate and choose storage
Accept the ImageNet terms first. The token is entered without echo and is never written to the notebook, config, log, or checkpoint.

In [ ]:
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face read token: ')
assert os.environ['HF_TOKEN'].startswith('hf_'), 'Expected a Hugging Face token'
print('token loaded for this runtime only')

In [ ]:
# Prefer the writable local filesystem with the most free space. Colab can
# expose large read-only mounts such as /kaggle/input, so df alone is unsafe.
preferred = [Path('/mnt/local-scratch'), Path('/content'), Path('/local_nvme'), Path('/tmp'), Path.home()]
df_lines = subprocess.check_output(['df', '-P', '-B1'], text=True).splitlines()[1:]
preferred.extend(Path(parts[5]) for line in df_lines if len(parts := line.split()) >= 6)
writable_disks, seen_devices = [], set()
for candidate in preferred:
    if str(candidate).startswith(('/content/drive', '/kaggle/input', '/proc', '/sys', '/dev')):
        continue
    try:
        if not candidate.is_dir():
            continue
        device = candidate.stat().st_dev
        if device in seen_devices:
            continue
        probe = candidate / f'.lsso-write-probe-{os.getpid()}'
        probe.mkdir(exist_ok=False)
        probe.rmdir()
        writable_disks.append((shutil.disk_usage(candidate).free, candidate))
        seen_devices.add(device)
    except (OSError, PermissionError):
        continue
assert writable_disks, 'No writable local filesystem was found'
SCRATCH_ROOT = max(writable_disks, key=lambda item: item[0])[1]
CACHE_ROOT = SCRATCH_ROOT / 'lsso-imagenet-wds'
OUTPUT_ROOT = COLAB_ROOT / 'LSSO-checkpoints' / 'imagenet1k'
CONFIG = {
    'model': 'vision_llama_base_rrlsso_r32',  # change to vision_llama_base_lsso_r32 for the paired run
    'cache_dir': str(CACHE_ROOT),
    'output': str(OUTPUT_ROOT / 'vision_llama_base_rrlsso_r32'),
    'epochs': 300,
    'batch_size': 768,
    'eval_batch_size': 768,
    'grad_accum': 1,
    'workers': 32,
    'eval_workers': 4,  # independent, non-persistent validation pool
    'max_downloads': 8,  # concurrent direct shard streams across loader workers
    'download_attempts': 0,  # retry transient network failures indefinitely
    'seed': 0,
}
Path(CONFIG['cache_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['output']).mkdir(parents=True, exist_ok=True)
# The data path follows Hugging Face's official WebDataset recipe: direct
# resolver streams feed the tar reader immediately and are cached in parallel.
# Clear settings left by older Xet-based notebook revisions.
os.environ.pop('HF_XET_HIGH_PERFORMANCE', None)
os.environ.pop('HF_XET_FIXED_DOWNLOAD_CONCURRENCY', None)
os.environ.setdefault('HF_HOME', str(CACHE_ROOT / '.hf-home'))
os.environ.setdefault('HF_XET_CACHE', str(CACHE_ROOT / '.xet'))
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '120')
print('selected scratch:', SCRATCH_ROOT)
print(json.dumps(CONFIG, indent=2))

## 3. Checks
Check CUDA, model registration, disk space, and gated repository access before spending GPU time.

In [ ]:
import torch, timm
import examples.models
import huggingface_hub
from huggingface_hub import HfApi

assert torch.cuda.is_available(), 'CUDA is unavailable'
assert CONFIG['model'] in timm.list_models('vision_llama_*')
HfApi(token=os.environ['HF_TOKEN']).dataset_info('timm/imagenet-1k-wds')
disk = os.statvfs(CONFIG['cache_dir'])
free_gb = disk.f_bavail * disk.f_frsize / 2**30
assert free_gb >= 180, f'Only {free_gb:.1f} GiB free; reserve at least 180 GiB'
gpu_name = torch.cuda.get_device_name()
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
compute_capability = torch.cuda.get_device_capability()
assert 'RTX PRO 6000' in gpu_name and compute_capability == (12, 0) and gpu_gb >= 90, (
    f'Expected RTX PRO 6000 Blackwell 96GB (SM120), got {gpu_name}, ' 
    f'SM{compute_capability[0]}{compute_capability[1]} ({gpu_gb:.1f} GiB)'
)
assert torch.version.cuda and torch.version.cuda.startswith(wanted_cuda), (
    f'Expected PyTorch CUDA {wanted_cuda}, got {torch.version.cuda}'
)
print(gpu_name, f'SM{compute_capability[0]}{compute_capability[1]},',
      f'{gpu_gb:.1f} GiB VRAM, {free_gb:.1f} GiB cache space free')
print('Hugging Face Hub:', huggingface_hub.__version__,
      '| WebDataset transport: direct stream-through cache')

### Build the fused SM120 backend
This downloads NVIDIA MathDx once per Colab runtime, compiles only for the allocated GPU, and fails loudly if the optimized backend cannot be loaded.

In [ ]:
import tarfile, urllib.request

if 'release 12.8' in nvcc_version:
    CUDA_PACKAGE, MATHDX_VERSION = 'cuda12', '25.12.1'
elif 'release 13.' in nvcc_version:
    CUDA_PACKAGE, MATHDX_VERSION = 'cuda13', '26.06.0'
else:
    raise RuntimeError(nvcc_version)
MATHDX_PACKAGE = f'nvidia-mathdx-{MATHDX_VERSION}-{CUDA_PACKAGE}'
MATHDX_CACHE = COLAB_ROOT / '.cache' / 'lsso-mathdx' / MATHDX_PACKAGE
MATHDX_ARCHIVE = MATHDX_CACHE / f'{MATHDX_PACKAGE}.tar.gz'
MATHDX_CACHE.mkdir(parents=True, exist_ok=True)
configs = list(MATHDX_CACHE.rglob('mathdx-config.cmake'))
if not configs:
    url = (
        'https://developer.nvidia.com/downloads/compute/cublasdx/redist/'
        f'cublasdx/{CUDA_PACKAGE}/{MATHDX_PACKAGE}.tar.gz'
    )
    if not MATHDX_ARCHIVE.is_file():
        print('downloading', url)
        urllib.request.urlretrieve(url, MATHDX_ARCHIVE)
    with tarfile.open(MATHDX_ARCHIVE, 'r:gz') as archive:
        archive.extractall(MATHDX_CACHE, filter='data')
    configs = list(MATHDX_CACHE.rglob('mathdx-config.cmake'))
assert len(configs) == 1, f'Expected one MathDx CMake config, found {configs}'
mathdx_root = configs[0].parents[3]
cuda_root = Path(nvcc).resolve().parents[1]
build_env = os.environ.copy()
build_env.update({
    'PYTHON_BIN': sys.executable,
    'MATHDX_ROOT': str(mathdx_root),
    'CUDA_HOME': str(cuda_root),
    'LSSO_CUDA_ARCHITECTURES': '120-real',
    'LSSO_MATHDX_LTO_ARCHITECTURES': '120',
    'LSSO_MATHDX_BUILD_DIR': str(MATHDX_CACHE / 'build-sm120'),
    'LSSO_MATHDX_RECONFIGURE': '1',
})
subprocess.run(['bash', 'tools/build_mathdx_backend.sh'], cwd=ROOT, env=build_env, check=True)
from lsso.mathdx_backend import is_mathdx_available, mathdx_load_error
assert is_mathdx_available(), f'MathDx backend failed to load: {mathdx_load_error()}'
print('MathDx backend: loaded (SM120 fused kernels enabled)')

In [ ]:
# Run real train/validation batches directly from incomplete remote shards.
# The helper forwards bytes to WebDataset immediately while caching them.
smoke_started = time.time()
smoke_output = str(Path(CONFIG['output']).with_name(Path(CONFIG['output']).name + '_smoke'))
smoke = [sys.executable, 'experiments/imagenet_wds_train.py',
         '--model', CONFIG['model'], '--cache-dir', CONFIG['cache_dir'],
         '--output', smoke_output, '--epochs', '1', '--steps-per-epoch', '2',
         '--max-val-steps', '2', '--batch-size', '8', '--eval-batch-size', '8',
         '--workers', '0', '--eval-workers', '0', '--max-downloads', str(CONFIG['max_downloads']),
         '--download-attempts', str(CONFIG['download_attempts']),
         '--shard-limit', '1', '--shuffle-buffer', '128',
         '--seed', str(CONFIG['seed']), '--require-mathdx', '--no-resume']
subprocess.run(smoke, check=True, env=os.environ.copy())
print(f'streaming smoke completed in {time.time() - smoke_started:.1f}s')

## 4. Launch detached training (or resume)
Run this cell once. It starts training in a new process session, writes `trainer.pid`, and returns immediately. Re-running it while the same trainer is alive is rejected. `--resume` automatically loads `last.pt` from the shallow checkpoint directory.

In [ ]:
output_dir = Path(CONFIG['output'])
output_dir.mkdir(parents=True, exist_ok=True)
pid_path = output_dir / 'trainer.pid'
log_path = output_dir / 'train.log'

def live_trainer(pid):
    try:
        state = Path(f'/proc/{pid}/stat').read_text().split()[2]
        command_line = Path(f'/proc/{pid}/cmdline').read_bytes().replace(b'\0', b' ').decode()
        return state != 'Z' and 'experiments/imagenet_wds_train.py' in command_line and str(output_dir) in command_line
    except (FileNotFoundError, PermissionError, ProcessLookupError, ValueError):
        return False

if pid_path.is_file():
    old_pid = int(pid_path.read_text().strip())
    if live_trainer(old_pid):
        raise RuntimeError(f'trainer PID {old_pid} is already running; do not launch twice')
    pid_path.unlink()

command = [sys.executable, '-u', 'experiments/imagenet_wds_train.py',
           '--model', CONFIG['model'], '--cache-dir', CONFIG['cache_dir'],
           '--output', CONFIG['output'], '--epochs', str(CONFIG['epochs']),
           '--batch-size', str(CONFIG['batch_size']),
           '--eval-batch-size', str(CONFIG['eval_batch_size']),
           '--grad-accum', str(CONFIG['grad_accum']),
           '--workers', str(CONFIG['workers']),
           '--eval-workers', str(CONFIG['eval_workers']),
           '--max-downloads', str(CONFIG['max_downloads']),
           '--download-attempts', str(CONFIG['download_attempts']),
           '--seed', str(CONFIG['seed']), '--require-mathdx', '--resume']
log = log_path.open('a', buffering=1)
process = subprocess.Popen(
    command, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy(),
    cwd=ROOT, start_new_session=True, close_fds=True,
)
log.close()
pid_path.write_text(str(process.pid))
time.sleep(3)
returncode = process.poll()
if returncode is not None:
    pid_path.unlink(missing_ok=True)
    subprocess.run(['tail', '-n', '40', str(log_path)], check=False)
    raise RuntimeError(f'trainer exited during startup with return code {returncode}')
print('detached trainer started')
print('pid:', process.pid)
print('output:', output_dir)
print('log:', log_path)
print('Run the independent monitor cell below; stopping it will not stop training.')

## 5. Independent 120-second monitor
This cell discovers the trainer from `/proc` rather than relying on the launcher object or PID file. Stop it whenever you need the kernel for archiving; stopping this monitor never signals the training process.

In [ ]:
from IPython.display import clear_output
monitor_output = Path(CONFIG['output'])
monitor_cache = Path(CONFIG['cache_dir'])

def find_trainers():
    trainers = []
    for proc_dir in Path('/proc').glob('[0-9]*'):
        try:
            command_line = proc_dir.joinpath('cmdline').read_bytes().replace(b'\0', b' ').decode(errors='replace')
            state = proc_dir.joinpath('stat').read_text().split()[2]
        except (FileNotFoundError, PermissionError, ProcessLookupError, ValueError):
            continue
        if 'experiments/imagenet_wds_train.py' in command_line and str(monitor_output) in command_line and state != 'Z':
            trainers.append((int(proc_dir.name), state))
    return trainers

try:
    while True:
        clear_output(wait=True)
        print(time.strftime('%Y-%m-%d %H:%M:%S'))
        trainers = find_trainers()
        print('trainers:', trainers if trainers else 'not found')
        subprocess.run([
            'nvidia-smi',
            '--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw',
            '--format=csv,noheader'], check=False)
        complete = list(monitor_cache.glob('*.tar'))
        partial = list(monitor_cache.glob('*.partial')) + list(monitor_cache.rglob('*.incomplete'))
        complete_gb = sum(path.stat().st_size for path in complete) / 2**30
        partial_gb = sum(path.stat().st_size for path in partial) / 2**30
        print(f'cached shards: {len(complete)} ({complete_gb:.1f} GiB); partial: {len(partial)} ({partial_gb:.1f} GiB)')
        print('--- metrics ---')
        subprocess.run(['tail', '-n', '6', str(monitor_output / 'metrics.csv')], check=False)
        print('--- log tail ---')
        subprocess.run(['tail', '-n', '20', str(monitor_output / 'train.log')], check=False)
        last = monitor_output / 'last.pt'
        if last.is_file():
            updated = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(last.stat().st_mtime))
            print(f'last.pt: {last.stat().st_size / 2**30:.2f} GiB, updated {updated}')
        else:
            print('last.pt: not created yet')
        print('next refresh in 120 seconds')
        for _ in range(24):
            time.sleep(5)
except KeyboardInterrupt:
    print('Monitoring stopped; the trainer was not signaled.')

## 6. Create and download a resume archive
Stop only the monitor cell, then run this cell while training continues. It takes a stable copy of `last.pt`, verifies that PyTorch can load it, packages the resume metadata, downloads one ZIP, and leaves the trainer untouched. Set `INCLUDE_BEST=True` for a final archive.

In [ ]:
from datetime import datetime
import gc, zipfile
from google.colab import files

run_dir = Path(CONFIG['output'])
last_source = run_dir / 'last.pt'
assert last_source.is_file(), 'last.pt is not available; finish at least one epoch first'
INCLUDE_BEST = False
archive_root = COLAB_ROOT / 'LSSO-archives'
archive_root.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
stage = archive_root / f'{run_dir.name}-{stamp}'
archive_path = archive_root / f'{run_dir.name}-{stamp}.zip'
stage.mkdir(parents=True, exist_ok=False)

def copy_stable(source, destination, attempts=10):
    for attempt in range(1, attempts + 1):
        before = source.stat()
        signature = (before.st_size, before.st_mtime_ns)
        shutil.copy2(source, destination)
        after = source.stat()
        if signature == (after.st_size, after.st_mtime_ns) and destination.stat().st_size == after.st_size:
            return
        destination.unlink(missing_ok=True)
        print(f'{source.name} changed during copy; retry {attempt}/{attempts}')
        time.sleep(3)
    raise RuntimeError(f'{source.name} remained unstable; retry after checkpoint writing finishes')

try:
    checkpoint_names = ['last.pt']
    if INCLUDE_BEST and (run_dir / 'best.pt').is_file():
        checkpoint_names.append('best.pt')
    for name in checkpoint_names:
        copy_stable(run_dir / name, stage / name)
    for name in ('config.json', 'metrics.csv', 'train.log'):
        if (run_dir / name).is_file():
            shutil.copy2(run_dir / name, stage / name)
    checkpoint = torch.load(stage / 'last.pt', map_location='cpu', weights_only=False, mmap=True)
    assert {'model', 'optimizer', 'epoch'} <= checkpoint.keys()
    print(f"validated epoch={checkpoint['epoch']} update={checkpoint.get('global_update')} best_acc={checkpoint.get('best_acc')}")
    del checkpoint
    gc.collect()
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as bundle:
        for source in sorted(stage.iterdir()):
            bundle.write(source, arcname=f'{run_dir.name}/{source.name}')
    with zipfile.ZipFile(archive_path) as bundle:
        assert bundle.testzip() is None, 'archive CRC validation failed'
finally:
    shutil.rmtree(stage, ignore_errors=True)
print(f'archive: {archive_path} ({archive_path.stat().st_size / 2**30:.2f} GiB)')
files.download(str(archive_path))

## 7. Paired experiment
After RRLSSO finishes, change `model` and `output` to `vision_llama_base_lsso_r32` and rerun the checks, smoke test, and detached launcher. Both runs reuse exactly the same cached ImageNet shards and augmentation recipe. The registered MHA entry is an official-compatible reproduction path, not a required new formal run.